In [1]:
import re
import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from data_utils import preprocessing

from tqdm import tqdm

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.tokenize.treebank import TreebankWordDetokenizer
from collections import Counter
import spacy
import re
from autocorrect import Speller
# from spellchecker import SpellChecker
import lightgbm as lgb
import nltk
nltk.download('punkt')
tqdm.pandas()

from sklearn.metrics import mean_squared_error


[nltk_data] Downloading package punkt to /home/sthueva/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [2]:
data_path = '../../input/commonlit-evaluate-student-summaries'
# os.listdir(data_path)

In [3]:
df = pd.read_csv(f'{data_path}/train_fold4_seed42_c1.csv')
print(df.shape)
display(df.head(2))

(7165, 9)


,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text,fold
0,000e8c3c7ddb,814d6b,The third wave was an experimentto see how peo...,0.205683,0.380538,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3
1,0070c9e7af47,814d6b,The Third Wave developed rapidly because the ...,3.272894,3.219757,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3


In [4]:
df['prompt_title_processed'] =  df['prompt_title'].apply(lambda x: preprocessing(x, "p4"))
df['prompt_question_processed'] =  df['prompt_question'].apply(lambda x: preprocessing(x, "p4"))
df['prompt_text_processed'] =  df['prompt_text'].apply(lambda x: preprocessing(x, "p4"))
df['text_processed'] =  df['text'].apply(lambda x: preprocessing(x, "p4"))

In [5]:
## Add preds

exp_name = "c1-442-d3lm21p4-512-8-22-5e51e3-C099050d0-rmse"
label_cols = ["content", "wording"]

folds = 4

preds_df=[]
for idx, fold in enumerate(range(folds)):
    print("Fold : ", fold)
    fold_df = pd.read_csv(f"../../output/commonlit-evaluate-student-summaries/{exp_name}/predictions_fold{fold}.csv")
    preds_df.append(fold_df)
    
preds_df = pd.concat(preds_df, axis=0)
preds_df.head()

Fold :  0
Fold :  1
Fold :  2
Fold :  3


,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text,fold,prompt_title_processed,prompt_question_processed,prompt_text_processed,text_processed,full_text_processed,full_text_len,content_predictions,wording_predictions
0,ad7245db300c,39c16e,The main person is a likable person that is ne...,-1.547163,-1.461245,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,Summarize at least 3 elements of an ideal trag...,Chapter 13 [BR] As the sequel to what has alre...,The main person is a likable person that is ne...,Content Wording [SEP] On Tragedy [Question] Su...,55,-1.481757,-1.436132
1,ac8891e90289,39c16e,a complex twisting plot that makes you feel pi...,-1.547163,-1.461245,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,Summarize at least 3 elements of an ideal trag...,Chapter 13 [BR] As the sequel to what has alre...,a complex twisting plot that makes you feel pi...,Content Wording [SEP] On Tragedy [Question] Su...,56,-1.517863,-1.606803
2,c4433d2e4905,39c16e,An ideal tragedy as described by Aristotle wou...,-1.072613,-0.999629,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,Summarize at least 3 elements of an ideal trag...,Chapter 13 [BR] As the sequel to what has alre...,An ideal tragedy as described by Aristotle wou...,Content Wording [SEP] On Tragedy [Question] Su...,56,-1.369738,-1.253718
3,551c0f2ac3de,39c16e,"An ideal tragedy must have a complex plot, ...",-0.681769,-1.672196,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,Summarize at least 3 elements of an ideal trag...,Chapter 13 [BR] As the sequel to what has alre...,"An ideal tragedy must have a complex plot, an ...",Content Wording [SEP] On Tragedy [Question] Su...,57,-1.340611,-1.298734
4,1732b9c702c1,39c16e,Aristole believed tragedies should follow a ha...,-1.264214,-1.505073,Summarize at least 3 elements of an ideal trag...,On Tragedy,Chapter 13 \r\nAs the sequel to what has alrea...,0,On Tragedy,Summarize at least 3 elements of an ideal trag...,Chapter 13 [BR] As the sequel to what has alre...,Aristole believed tragedies should follow a ha...,Content Wording [SEP] On Tragedy [Question] Su...,57,-1.402170,-1.481686


In [6]:
df["content_preds"] = df["student_id"].map(preds_df.set_index('student_id')["content_predictions"])
df["wording_preds"] = df["student_id"].map(preds_df.set_index('student_id')["wording_predictions"])

In [10]:
STOP_WORDS = set(stopwords.words('english'))

def word_overlap_count(row):
    """ intersection(prompt_text, text) """        
    def check_is_stop_word(word):
        return word in STOP_WORDS

    prompt_words = row['prompt_tokens']
    summary_words = row['text_tokens']
    if STOP_WORDS:
        prompt_words = list(filter(check_is_stop_word, prompt_words))
        summary_words = list(filter(check_is_stop_word, summary_words))
    return len(set(prompt_words).intersection(set(summary_words)))


def quotes_count(row):
        summary = row['text_processed']
        text = row['prompt_text_processed']
        quotes_from_summary = re.findall(r'"([^"]*)"', summary)
        if len(quotes_from_summary)>0:
            return [quote in text for quote in quotes_from_summary].count(True)
        else:
            return 0

In [11]:
# Feature Engineering
df["prompt_tokens"] = df["prompt_text_processed"].progress_apply(lambda x: word_tokenize(x))
df["text_tokens"] = df["text_processed"].progress_apply(lambda x: word_tokenize(x))
df["prompt_text_length"] = df["prompt_tokens"].progress_apply(lambda x: len(x))
df["text_length"] = df["text_tokens"].progress_apply(lambda x: len(x))
df["length_ratio"] = df['text_length'] / df['prompt_text_length']
df['quotes_count'] = df.progress_apply(quotes_count, axis=1)

# df['word_overlap_count'] = df.progress_apply(word_overlap_count, axis=1)


100%|██████████| 7165/7165 [00:00<00:00, 107039.09it/s]


In [12]:
df.head()

,student_id,prompt_id,text,content,wording,prompt_question,prompt_title,prompt_text,fold,prompt_title_processed,...,prompt_text_processed,text_processed,content_preds,wording_preds,prompt_tokens,text_tokens,prompt_text_length,text_length,length_ratio,quotes_count
0,000e8c3c7ddb,814d6b,The third wave was an experimentto see how peo...,0.205683,0.380538,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3,The Third Wave,...,Background [BR] The Third Wave experiment took...,The third wave was an experimentto see how peo...,0.052114,0.892328,"[Background, [, BR, ], The, Third, Wave, exper...","[The, third, wave, was, an, experimentto, see,...",678,64,0.094395,0
1,0070c9e7af47,814d6b,The Third Wave developed rapidly because the ...,3.272894,3.219757,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3,The Third Wave,...,Background [BR] The Third Wave experiment took...,The Third Wave developed rapidly because the s...,1.722007,2.244170,"[Background, [, BR, ], The, Third, Wave, exper...","[The, Third, Wave, developed, rapidly, because...",678,232,0.342183,4
2,0095993991fe,814d6b,The third wave only started as an experiment w...,0.205683,0.380538,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3,The Third Wave,...,Background [BR] The Third Wave experiment took...,The third wave only started as an experiment w...,0.085442,0.854999,"[Background, [, BR, ], The, Third, Wave, exper...","[The, third, wave, only, started, as, an, expe...",678,67,0.098820,1
3,00c20c6ddd23,814d6b,The experimen was orginally about how even whe...,0.567975,0.969062,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3,The Third Wave,...,Background [BR] The Third Wave experiment took...,The experimen was orginally about how even whe...,0.342642,1.218832,"[Background, [, BR, ], The, Third, Wave, exper...","[The, experimen, was, orginally, about, how, e...",678,86,0.126844,0
4,00d40ad10dc9,814d6b,The third wave developed so quickly due to the...,-0.910596,-0.081769,Summarize how the Third Wave developed over su...,The Third Wave,Background \r\nThe Third Wave experiment took ...,3,The Third Wave,...,Background [BR] The Third Wave experiment took...,The third wave developed so quickly due to the...,-0.946387,-0.155389,"[Background, [, BR, ], The, Third, Wave, exper...","[The, third, wave, developed, so, quickly, due...",678,29,0.042773,0


In [15]:
targets = ["content", "wording"]

drop_columns = ["fold", "student_id", "prompt_id", "text", 
                "prompt_question", "prompt_title", "prompt_text", "prompt_tokens", "text_tokens",
                "prompt_question_processed", "prompt_title_processed", "prompt_text_processed", "text_processed"
               ] + targets

In [20]:
model_dict = {}

for target in targets:
    models = []
    
    for fold in range(df.fold.nunique()):
        if target == "content":
            rem_cols = drop_columns + ["wording_preds"]
        else:
            rem_cols = drop_columns + ["content_preds"]

        X_train_cv = df[df["fold"] != fold].drop(columns=rem_cols)
        print(X_train_cv.columns)
        y_train_cv = df[df["fold"] != fold][target]

        X_eval_cv = df[df["fold"] == fold].drop(columns=rem_cols)
        y_eval_cv = df[df["fold"] == fold][target]

        dtrain = lgb.Dataset(X_train_cv, label=y_train_cv)
        dval = lgb.Dataset(X_eval_cv, label=y_eval_cv)

        params = {
            'boosting_type': 'gbdt',
            'random_state': 42,
            'objective': 'regression',
            'metric': 'rmse',
            'learning_rate': 0.048,
            'max_depth': 3,  #3
            'lambda_l1': 0.0,
            'lambda_l2': 0.011
        }

        evaluation_results = {}
        model = lgb.train(params,
                          num_boost_round=1000,
                            #categorical_feature = categorical_features,
                          valid_names=['train', 'valid'],
                          train_set=dtrain,
                          valid_sets=dval,
                          callbacks=[
                              lgb.early_stopping(stopping_rounds=30, verbose=True),
                               lgb.log_evaluation(100),
                              lgb.callback.record_evaluation(evaluation_results)
                            ],
                          )
        models.append(model)
    
    model_dict[target] = models

Index(['content_preds', 'prompt_text_length', 'text_length', 'length_ratio',
       'quotes_count'],
      dtype='object')
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000175 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 777
[LightGBM] [Info] Number of data points in the train set: 5108, number of used features: 5
[LightGBM] [Info] Start training from score 0.017606
Training until validation scores don't improve for 30 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fur

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Early stopping, best iteration is:
[89]	train's rmse: 0.674874


In [21]:
# cv
rmses = []

for target in targets:
    models = model_dict[target]

    preds = []
    trues = []
    
    for fold, model in enumerate(models):
        if target == "content":
            rem_cols = drop_columns + ["wording_preds"]
        else:
            rem_cols = drop_columns + ["content_preds"]
        X_eval_cv = df[df["fold"] == fold].drop(columns=rem_cols)
        y_eval_cv = df[df["fold"] == fold][target]

        pred = model.predict(X_eval_cv)

        trues.extend(y_eval_cv)
        preds.extend(pred)
        
    rmse = np.sqrt(mean_squared_error(trues, preds))
    print(f"{target}_rmse : {rmse}")
    rmses = rmses + [rmse]

print(f"mcrmse : {sum(rmses) / len(rmses)}")

content_rmse : 0.49472366096430376
wording_rmse : 0.5972802998142142
mcrmse : 0.546001980389259


In [ ]:
# before merge preprocess
df["prompt_tokens"] = df["prompt_text_processed"].apply(lambda x: word_tokenize(x))

df["summary_length"] = df["text_processed"].apply(lambda x: len(word_tokenize(x)))
df["summary_tokens"] = df["text_processed"].apply(lambda x: word_tokenize(x))

# Add prompt tokens into spelling checker dictionary
prompts["prompt_tokens"].apply(lambda x: self.add_spelling_dictionary(x))
        
#         from IPython.core.debugger import Pdb; Pdb().set_trace()
        # fix misspelling
        summaries["fixed_summary_text"] = summaries["text"].progress_apply(
            lambda x: self.speller(x)
        )
        
        # count misspelling
        summaries["splling_err_num"] = summaries["text"].progress_apply(self.spelling)
        
        # merge prompts and summaries
        input_df = summaries.merge(prompts, how="left", on="prompt_id")

        
        input_df['word_overlap_count'] = input_df.progress_apply(self.word_overlap_count, axis=1)
        input_df['bigram_overlap_count'] = input_df.progress_apply(
            self.ngram_co_occurrence,args=(2,), axis=1 
        )
        input_df['bigram_overlap_ratio'] = input_df['bigram_overlap_count'] / (input_df['summary_length'] - 1)
        
        input_df['trigram_overlap_count'] = input_df.progress_apply(
            self.ngram_co_occurrence, args=(3,), axis=1
        )
        input_df['trigram_overlap_ratio'] = input_df['trigram_overlap_count'] / (input_df['summary_length'] - 2)
        
        input_df['quotes_count'] = input_df.progress_apply(self.quotes_count, axis=1)
        
        return input_df.drop(columns=["summary_tokens", "prompt_tokens"])

In [ ]:
df["prompt_title"].unique()

In [ ]:
df["prompt_question"].unique()

In [ ]:
plot_cols = ["content", "wording"]

rows=1
cols=len(plot_cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*3))

for idx, col in enumerate(plot_cols):
    sns.histplot(x=col, data=df, ax=axes[idx])

In [ ]:
plot_cols = ["content", "wording"]

rows=df.prompt_id.nunique()
cols=len(plot_cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*3))

for row_idx, prompt_id  in enumerate(df.prompt_id.unique()):
    p_df = df[df['prompt_id']==prompt_id]
    for idx, col in enumerate(plot_cols):
        sns.histplot(x=col, data=p_df, ax=axes[row_idx, idx], hue="prompt_id")

In [ ]:
df["prompt_text"].unique()

In [ ]:
df['full_text'] = df["prompt_title"] + " " + df["prompt_text"]+ " " + df["prompt_question"]+ " " + df["text"]
df['full_text_processed'] =  df['full_text'].apply(lambda x: preprocessing(x))

df['full_text_len'] = df['full_text_processed'].apply(lambda x: len(x.split()))
df['unique_word_len'] = df['full_text_processed'].apply(lambda x: len(set(x.split())))

print(f"Min length in full text: {df['full_text_len'].min()}")
print(f"Max length in full text: {df['full_text_len'].max()}")
print(f"Number of samples longer than max length: {df[df['full_text_len']> 768].shape}")

In [ ]:
plot_cols = ["content", "wording", "full_text_len", "unique_word_len"]

for col in plot_cols:
    df[f"{col}_log"] = np.log(df[col]+1)
    df[f"{col}_sqrt"] = np.sqrt(df[col])
    df[f"{col}_cbrt"] = np.cbrt(df[col])
    if col != "pixel_count":
        df[f"{col}_power"] = power_transform(df[col].values.reshape(-1, 1)).squeeze(-1)

In [ ]:
rows=5
cols=len(plot_cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*3))

for idx, col in enumerate(plot_cols):
    sns.histplot(x=col, data=df, ax=axes[0, idx])
    sns.histplot(x=f"{col}_log", data=df, ax=axes[1, idx])
    sns.histplot(x=f"{col}_sqrt", data=df, ax=axes[2, idx])
    sns.histplot(x=f"{col}_cbrt", data=df, ax=axes[3, idx])
    sns.histplot(x=f"{col}_power", data=df, ax=axes[4, idx])

In [ ]:
plot_cols = ["content", "wording", "full_text_len", "unique_word_len"]

rows=df.prompt_id.nunique()
cols=len(plot_cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*3))

for row_idx, prompt_id in enumerate(df.prompt_id.unique()):
    prom_df = df[df["prompt_id"] == prompt_id]
    for idx, col in enumerate(plot_cols):
        sns.histplot(x=col, data=prom_df, ax=axes[row_idx, idx])

In [ ]:
plot_cols = ["content", "wording", "full_text_len"]

rows=1
cols=len(plot_cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols*5, rows*3))

for idx, col in enumerate(plot_cols):
    sns.histplot(x=col, data=df, ax=axes[idx])

In [ ]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('microsoft/deberta-v3-base')

In [ ]:
# tokenizer.add_special_tokens({'additional_special_tokens': ['[BR]', '[C]', '[W]']})

In [ ]:
tokenizer.all_special_tokens

In [ ]:
dir(tokenizer)

In [ ]:
tokenizer.all_special_ids

In [ ]:
tokenizer.encode??


In [ ]:
[9] * 4

In [ ]:
t = tokenizer(["[CLS][C][W]", 
               "[CLS] yes"],
            padding=True, 

             return_tensors='pt')

In [ ]:
t['input_ids'].shape

In [ ]:
tokenizer.tokenize("[CLS] content wording the text is perfect \n\n yes ")

In [ ]:
tokenizer.tokenize("[CLS][C][W] the text is perfect")

In [ ]:
tokenizer.tokenize(df.loc[0]["full_text_processed"])

In [ ]:
df["text"].unique()[0], df["content"].unique()[0], df["wording"].unique()[0]

In [ ]:
df["text"].unique()[10],  df["content"].unique()[10], df["wording"].unique()[10]

In [ ]:
df["text"].unique()[100],  df["content"].unique()[100], df["wording"].unique()[100]

In [ ]:
import string
string.punctuation

In [ ]:
s= df["prompt_text"][0].translate(str.maketrans('', '', string.punctuation))
s

In [ ]:
pip install datasets

In [ ]:
from datasets import Dataset
train_set = Dataset.from_pandas(df)


In [ ]:
train_set

In [ ]:
type(train_set)

In [ ]:
def preprocess(example):
    first_sentence = [example['prompt']] * 5
    second_sentences = [example[option] if example[option] is not None else '-' for option in 'ABCDE']
    tokenized_example = tokenizer(first_sentence, second_sentences, truncation=True)
    tokenized_example['label'] = option_to_index[example['answer']]
    
    return tokenized_example

In [ ]:
train_dataset = train_set.map(preprocess, remove_columns=['prompt', 'A', 'B', 'C', 'D', 'E', 'answer'])